In [1]:
import os
import sys
import importlib
from src.load_blocks import load_blocks_from_ckpt_implicit

# Force reload of the modules to get the latest changes
if 'src.implicit_llt' in sys.modules:
    importlib.reload(sys.modules['src.implicit_llt'])
if 'src.load_blocks' in sys.modules:
    importlib.reload(sys.modules['src.load_blocks'])



ckpt_path = os.path.join(os.getcwd(), "artifacts", "models", "3d_blocks.pt")

blocks = load_blocks_from_ckpt_implicit(ckpt_path, device="cpu")  # or "cuda"/"cpu"

cl_block_wing = blocks["wing"]["cl_block"]
cd_block_wing = blocks["wing"]["cd_block"]
cm_block_wing = blocks["wing"]["cm_block"]

# Verify requires_grad is True
print("Wing shape params requires_grad:", [p.requires_grad for p in blocks["wing"]["shape_params"]])
print("Elevator shape params requires_grad:", [p.requires_grad for p in blocks["elevator"]["shape_params"]])

Wing shape params requires_grad: [True, True, True, True]
Elevator shape params requires_grad: [True, True, True, True]


In [2]:
# Better diagnostic cell
import torch

print("=== Parameter Check ===")
for i, p in enumerate(blocks["wing"]["shape_params"]):
    print(f"Param {i}: shape={p.shape}, mean={p.mean().item():.6f}, range=[{p.min().item():.6f}, {p.max().item():.6f}]")

print("\n=== CL Evaluation Check ===")
alpha_test, V_test = 5.0, 18.0
CL_test = cl_block_wing(alpha_test, V_test)
print(f"CL({alpha_test}°, {V_test} m/s) = {CL_test}")

# Check if CL is reasonable (should be ~0.5 for a wing at 5 degrees)
if abs(CL_test) > 10:
    print("⚠️  WARNING: CL value is unusually large!")
if abs(CL_test) < 0.01:
    print("⚠️  WARNING: CL value is unusually small!")

print("\n=== Gradient Magnitude Check ===")
grads_test = cl_block_wing.backward(alpha_test, V_test)
for i, g in enumerate(grads_test):
    if g is not None:
        print(f"Grad {i}: mean={g.abs().mean().item():.2e}, max={g.abs().max().item():.2e}")

=== Parameter Check ===
Param 0: shape=torch.Size([8]), mean=0.150273, range=[0.126376, 0.173763]
Param 1: shape=torch.Size([8]), mean=-0.150273, range=[-0.173763, -0.126376]
Param 2: shape=torch.Size([]), mean=-0.000000, range=[-0.000000, -0.000000]
Param 3: shape=torch.Size([]), mean=0.002548, range=[0.002548, 0.002548]

=== CL Evaluation Check ===
CL(5.0°, 18.0 m/s) = 0.5307552218437195

=== Gradient Magnitude Check ===
Grad 0: mean=2.77e-01, max=4.09e-01
Grad 1: mean=1.36e-01, max=2.22e-01
Grad 2: mean=3.68e-02, max=3.68e-02
Grad 3: mean=2.83e+00, max=2.83e+00


In [3]:
# Test with the OLD non-implicit version
from src.load_blocks import load_blocks_from_ckpt

blocks_old = load_blocks_from_ckpt(ckpt_path, device="cpu")
cl_block_old = blocks_old["wing"]["cl_block"]

CL_old = cl_block_old(5.0, 18.0)
print(f"OLD (non-implicit) CL: {CL_old}")
print(f"NEW (implicit) CL: {CL_test}")

OLD (non-implicit) CL: 0.5307552218437195
NEW (implicit) CL: 0.5307552218437195


In [4]:
import torch
ckpt = torch.load(ckpt_path, map_location="cpu")
print("Checkpoint metadata:")
print(ckpt.get("meta", {}))

Checkpoint metadata:
{}


In [5]:
alpha, V = 5.0, 18.0
CL = cl_block_wing(alpha, V)
grads = cl_block_wing.backward(alpha, V)

In [6]:
print(f"CL at alpha={alpha} deg, V={V} m/s: {CL}")
print(f"CL value type: {type(CL)}")
print(f"CL magnitude: {abs(CL) if isinstance(CL, (int, float)) else abs(CL.item())}")


CL at alpha=5.0 deg, V=18.0 m/s: 0.5307552218437195
CL value type: <class 'float'>
CL magnitude: 0.5307552218437195


In [7]:
print(f"CL at alpha={alpha} deg, V={V} m/s: {CL}")
print(f"Gradients: {grads}\n")

CL at alpha=5.0 deg, V=18.0 m/s: 0.5307552218437195
Gradients: [tensor([-0.0757,  0.1350,  0.3600,  0.4093,  0.3311,  0.2640,  0.2928,  0.3514]), tensor([ 0.1488,  0.2065,  0.2224,  0.1295, -0.0564, -0.1822, -0.1384, -0.0007]), tensor(0.0368), tensor(-2.8299)]



# Gradient sanity check

In [8]:
import os
import torch
import numpy as np

from src.load_blocks import load_blocks_from_ckpt_implicit

device_set= "cpu"

# 1) Load differentiable blocks
ckpt_path = os.path.join(os.getcwd(), "artifacts", "models", "3d_blocks.pt")
blocks = load_blocks_from_ckpt_implicit(ckpt_path, device=device_set)  # or "cuda"/"cpu"
cl_block_wing_implicit = blocks["wing"]["cl_block"]
shape_params_wing = blocks["wing"]["shape_params"]

# 2) Pick a test point
alpha0, V0 = 5.0, 18.0

# 3) Get autograd gradient wrt first Kulfan parameter (as an example)
#    (You can sum all params into one scalar “direction” to compare.)
for p in shape_params_wing:
    if p.grad is not None:
        p.grad.zero_()

alpha_t = torch.tensor(alpha0, dtype=torch.float32, device=device_set)
V_t     = torch.tensor(V0, dtype=torch.float32, device=device_set)

# forward
CL_t = cl_block_wing_implicit.llt_model(alpha_t, V_t)["CL"]
CL_scalar = CL_t.sum()  # ensure scalar for backward

# backward: gradients wrt all shape params
CL_scalar.backward()

autograd_grads = [p.grad.detach().clone().cpu().numpy() for p in shape_params_wing]

# 4) Finite-difference check on a single parameter entry
eps = 1e-4
param_idx = 0      # which tensor in shape_params_wing
elem_idx  = 0      # which element inside that tensor

p = shape_params_wing[param_idx]
p_data = p.detach().clone()

# helper: evaluate CL with current params
def eval_CL():
    with torch.no_grad():
        return float(cl_block_wing_implicit(alpha0, V0))

CL0 = eval_CL()

# +eps
p.data = p_data.clone()
p.data.view(-1)[elem_idx] += eps
CL_plus = eval_CL()

# -eps
p.data = p_data.clone()
p.data.view(-1)[elem_idx] -= eps
CL_minus = eval_CL()

# restore original param
p.data = p_data

fd_grad = (CL_plus - CL_minus) / (2 * eps)
auto_grad = autograd_grads[param_idx].reshape(-1)[elem_idx]

print("Finite-diff grad dCL/dp:", fd_grad)
print("Autograd grad dCL/dp:   ", auto_grad)
print("Relative error:", abs(fd_grad - auto_grad) / (abs(fd_grad) + 1e-12))


Finite-diff grad dCL/dp: -0.07539987564086914
Autograd grad dCL/dp:    -0.07572059
Relative error: 0.0042535573121965506


In [9]:
import os
import torch
import numpy as np

from src.load_blocks import load_blocks_from_ckpt
device_set= "cpu"
# 1) Load differentiable blocks
ckpt_path = os.path.join(os.getcwd(), "artifacts", "models", "3d_blocks.pt")
blocks = load_blocks_from_ckpt(ckpt_path, device=device_set)  # or "cuda"/"cpu"
cl_block_wing_explicit = blocks["wing"]["cl_block"]
shape_params_wing = blocks["wing"]["shape_params"]

# 2) Pick a test point
alpha0, V0 = 5.0, 18.0

# 3) Get autograd gradient wrt first Kulfan parameter (as an example)
#    (You can sum all params into one scalar “direction” to compare.)
for p in shape_params_wing:
    if p.grad is not None:
        p.grad.zero_()

alpha_t = torch.tensor(alpha0, dtype=torch.float32, device=device_set)
V_t     = torch.tensor(V0, dtype=torch.float32, device=device_set)

# forward
CL_t = cl_block_wing_explicit.llt_model(alpha_t, V_t)["CL"]
CL_scalar = CL_t.sum()  # ensure scalar for backward

# backward: gradients wrt all shape params
CL_scalar.backward()

autograd_grads = [p.grad.detach().clone().cpu().numpy() for p in shape_params_wing]

# 4) Finite-difference check on a single parameter entry
eps = 1e-4
param_idx = 0      # which tensor in shape_params_wing
elem_idx  = 0      # which element inside that tensor

p = shape_params_wing[param_idx]
p_data = p.detach().clone()

# helper: evaluate CL with current params
def eval_CL():
    with torch.no_grad():
        return float(cl_block_wing_explicit(alpha0, V0))

CL0 = eval_CL()

# +eps
p.data = p_data.clone()
p.data.view(-1)[elem_idx] += eps
CL_plus = eval_CL()

# -eps
p.data = p_data.clone()
p.data.view(-1)[elem_idx] -= eps
CL_minus = eval_CL()

# restore original param
p.data = p_data

fd_grad = (CL_plus - CL_minus) / (2 * eps)
auto_grad = autograd_grads[param_idx].reshape(-1)[elem_idx]

print("Finite-diff grad dCL/dp:", fd_grad)
print("Autograd grad dCL/dp:   ", auto_grad)
print("Relative error:", abs(fd_grad - auto_grad) / (abs(fd_grad) + 1e-12))


Finite-diff grad dCL/dp: -0.07539987564086914
Autograd grad dCL/dp:    -0.07573207
Relative error: 0.004405731225238011


# Timing

In [12]:
import time
import torch
import numpy as np

def _sync(device):
    device = str(device)
    if "cuda" in device:
        torch.cuda.synchronize()
    elif "mps" in device:
        if hasattr(torch, "mps") and hasattr(torch.mps, "synchronize"):
            torch.mps.synchronize()
        else:
            torch.empty(1, device="mps").sum().item()

def _zero_grads(params):
    for p in params:
        if p.grad is not None:
            p.grad.zero_()

def time_block_tensor(
    block,
    alpha=5.0,
    V=18.0,
    n_warmup=10,
    n_repeat=30,
    name=""
):
    """
    Times:
      - forward: llt_model forward only (Tensor output)
      - backward: backward only (using the SAME forward tensor)
      - fwd+bwd: forward + backward
    This avoids the wrapper block.backward() which re-runs forward.
    """

    # infer device from params
    dev = block.params[0].device if hasattr(block, "params") and len(block.params) else torch.device("cpu")

    # Make inputs as tensors on the same device
    alpha_t = torch.tensor(alpha, dtype=torch.float32, device=dev)
    V_t     = torch.tensor(V, dtype=torch.float32, device=dev)

    # --------------------
    # Warm-up (un-timed)
    # --------------------
    for _ in range(n_warmup):
        _zero_grads(block.params)
        y = block.llt_model(alpha_t, V_t)["CL"].sum()
        y.backward()
    _sync(dev)

    # --------------------
    # Forward-only timing
    # --------------------
    f_times = []
    with torch.no_grad():  # forward-only timing doesn't need a graph
        for _ in range(n_repeat):
            _sync(dev)
            t0 = time.perf_counter()
            y = block.llt_model(alpha_t, V_t)["CL"]
            _sync(dev)
            f_times.append(time.perf_counter() - t0)

    # --------------------
    # Backward-only timing
    # --------------------
    # Important: we must build a graph for backward timing, so no no_grad here.
    b_times = []
    for _ in range(n_repeat):
        _zero_grads(block.params)
        y = block.llt_model(alpha_t, V_t)["CL"].sum()  # build graph
        _sync(dev)

        t0 = time.perf_counter()
        y.backward()
        _sync(dev)

        b_times.append(time.perf_counter() - t0)

    # --------------------
    # Forward + backward timing
    # --------------------
    fb_times = []
    for _ in range(n_repeat):
        _zero_grads(block.params)
        _sync(dev)

        t0 = time.perf_counter()
        y = block.llt_model(alpha_t, V_t)["CL"].sum()
        y.backward()
        _sync(dev)

        fb_times.append(time.perf_counter() - t0)

    def stats(ts):
        ts = np.array(ts, dtype=float)
        return dict(
            mean=float(ts.mean()),
            median=float(np.median(ts)),
            min=float(ts.min()),
            max=float(ts.max()),
        )

    print(f"\n[{name}] device={dev}")
    print("  forward  :", stats(f_times))
    print("  backward :", stats(b_times))
    print("  fwd+bwd  :", stats(fb_times))

    return f_times, b_times, fb_times

# explicit (unrolled)
time_block_tensor(cl_block_wing_explicit, alpha=5.0, V=18.0, name="EXPLICIT / unrolled")

# implicit (IFT)
time_block_tensor(cl_block_wing_implicit, alpha=5.0, V=18.0, name="IMPLICIT / IFT")



[EXPLICIT / unrolled] device=cpu
  forward  : {'mean': 0.05038114180060802, 'median': 0.04809010350436438, 'min': 0.04443249400355853, 'max': 0.08495932001096662}
  backward : {'mean': 0.023966106599740064, 'median': 0.02205396449426189, 'min': 0.020424870002898388, 'max': 0.06768386700423434}
  fwd+bwd  : {'mean': 0.10333898779839122, 'median': 0.10185881749202963, 'min': 0.09599962799984496, 'max': 0.11665406300744507}

[IMPLICIT / IFT] device=cpu
  forward  : {'mean': 0.04008787479882206, 'median': 0.03958905499894172, 'min': 0.03744016500422731, 'max': 0.04537436499958858}
  backward : {'mean': 0.022739944701606875, 'median': 0.021868725496460684, 'min': 0.01997571399260778, 'max': 0.02802698600862641}
  fwd+bwd  : {'mean': 0.06365940736674626, 'median': 0.06197388700093143, 'min': 0.05581877600343432, 'max': 0.08362862200010568}


([0.040376613003900275,
  0.03840400399349164,
  0.041058428992982954,
  0.03856380798970349,
  0.03960249999363441,
  0.040646610999829136,
  0.039306004997342825,
  0.03744016500422731,
  0.04001229500863701,
  0.040412178001133725,
  0.03978229900531005,
  0.03845852900121827,
  0.04054593699402176,
  0.04147595600807108,
  0.0390379180025775,
  0.03864624998823274,
  0.03948304199730046,
  0.03817141099716537,
  0.03945224199560471,
  0.04424591199494898,
  0.04537436499958858,
  0.03942819399526343,
  0.03957561000424903,
  0.04011673999775667,
  0.042702070000814274,
  0.0426030200032983,
  0.03899710200494155,
  0.03879590399446897,
  0.040498453003237955,
  0.039422681991709396],
 [0.025933046999853104,
  0.023116355994716287,
  0.0214996529975906,
  0.024608075007563457,
  0.021725038008298725,
  0.02802698600862641,
  0.0278044709993992,
  0.022508662004838698,
  0.021057101999758743,
  0.025669299997389317,
  0.021493503008969128,
  0.021064337008283474,
  0.0209789349901257